# Step 1: Install jules

In [ ]:
!npm install -g @google/jules

#Step 2: Run the cell below, login via your browser and copy the token and paste it in the input box below.

In [ ]:
!jules login --no-launch-browser

#Step 3: Export sessions

In [ ]:
#!/usr/bin/env python3
"""
Jules data exporter (REST-only, Nov 2026)

What it exports:
  - sources.json          (all connected repos)
  - sessions.json         (list of all tasks/sessions)
  - sessions/<id>/        (per-session detail)
      session-slim.json   (title, prompt, plan, status, outputs summary)
      patch-*.diff        (code patches when present)
      summary.md          (human-readable overview)
  - repos/<sourceId>/     (setup.sh / test-command.txt recovered from sessions)
  - export-summary.json

What is currently unavailable (404):
  - user settings
  - environment variables
  - scheduled tasks
  - memories / knowledge (ListMemories / c77bjc)

Usage:
  python3 export_jules.py
"""

import json
import pathlib
import re
import sys
import time
import urllib.error
import urllib.parse
import urllib.request

def get_token() -> str:
    p = pathlib.Path.home() / ".jules/cache/oauth_creds.json"
    if p.exists():
        data = json.loads(p.read_text())
        return data["access_token"]
    sys.exit(
        "No credentials found.\n"
        "Run `jules login` first, or place a valid oauth_creds.json at:\n"
        f"  {p}"
    )

TOKEN = get_token()
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}
REST = "https://aida.googleapis.com/v1/swebot"
OUT = pathlib.Path("./jules-export")
OUT.mkdir(exist_ok=True)

errors: list[str] = []

def get(path: str, **params):
    q = ("?" + urllib.parse.urlencode(params)) if params else ""
    url = f"{REST}{path}{q}"
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read().decode() or "{}")

def paged(path: str, key: str, page_size: int = 100):
    items = []
    token = None
    while True:
        params = {"pageSize": page_size}
        if token:
            params["pageToken"] = token
        try:
            resp = get(path, **params)
        except urllib.error.HTTPError as e:
            errors.append(f"{path}: HTTP {e.code}")
            break
        items.extend(resp.get(key, []))
        token = resp.get("nextPageToken")
        if not token:
            break
        time.sleep(0.15)
    return items

def safe(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]", "_", str(s))

def dump(path: pathlib.Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))

print("=== Jules Export (REST) ===")
print(f"Output → {OUT.resolve()}\n")

# ------------------------------------------------------------------
# 1. Sources
# ------------------------------------------------------------------
print("Fetching sources...")
sources = paged("/sources", "sources")
dump(OUT / "sources.json", sources)
print(f"  {len(sources)} sources")

exit(0)



# ------------------------------------------------------------------
# 2. Sessions list
# ------------------------------------------------------------------
print("Fetching sessions list...")
sessions = paged("/tasks", "tasks", page_size=100)
dump(OUT / "sessions.json", sessions)
print(f"  {len(sessions)} sessions")

# ------------------------------------------------------------------
# 3. Per-session detail + patches
# ------------------------------------------------------------------
print("Processing individual sessions (this takes a while)...")
setup: dict = {}
patches = 0
processed = 0

for s in sorted(sessions, key=lambda x: x.get("createdAt", "")):
    tid = s.get("id")
    if not tid:
        continue

    d = OUT / "sessions" / str(tid)
    try:
        task_resp = get(f"/tasks/{tid}")
        task = task_resp.get("task", task_resp)
    except Exception as e:
        errors.append(f"session {tid}: {e}")
        continue

    # Slim session record (avoids huge files)
    slim = {
        "id": task.get("id"),
        "title": task.get("title") or s.get("suggestedTitle"),
        "sourceId": task.get("sourceId"),
        "taskStatus": task.get("taskStatus"),
        "createdAt": task.get("createdAt"),
        "completedAt": task.get("completedAt"),
        "description": (task.get("description") or s.get("description") or "")[:4000],
        "plan_steps": [
            p.get("title") for p in (task.get("latestPlan") or {}).get("steps", [])
        ],
        "outputs": [],
    }

    # Extract patches + setup scripts
    for o in task.get("outputs", []):
        if not isinstance(o, dict) or "gitCommit" not in o:
            continue
        c = o["gitCommit"]
        unidiff = (c.get("patch") or {}).get("unidiff")
        has_patch = bool(unidiff)
        if has_patch:
            j = len(list(d.glob("patch-*.diff"))) + 1
            d.mkdir(parents=True, exist_ok=True)
            (d / f"patch-{j}.diff").write_text(unidiff)
            patches += 1

        meta = c.get("repoMetadata") or {}
        if meta.get("environmentSetupScript") or meta.get("testCommand"):
            key = task.get("sourceId") or meta.get("name") or tid
            setup[key] = meta

        slim["outputs"].append({
            "title": (c.get("title") or (c.get("commitMessage") or "")[:100]),
            "branch": c.get("gitBranchName"),
            "has_patch": has_patch,
        })

    dump(d / "session-slim.json", slim)

    # Human-readable summary
    lines = [
        f"# {slim.get('title') or tid}",
        "",
        f"- **ID:** {tid}",
        f"- **Repo:** {slim.get('sourceId', '')}",
        f"- **Status:** {slim.get('taskStatus', '')}",
        f"- **Created:** {slim.get('createdAt', '')}",
        f"- **Completed:** {slim.get('completedAt', '')}",
        "",
        "## Prompt",
        "",
        slim.get("description") or "",
        "",
    ]
    if slim["plan_steps"]:
        lines += ["## Plan", ""] + [f"{i+1}. {t}" for i, t in enumerate(slim["plan_steps"])] + [""]
    if slim["outputs"]:
        lines += ["## Code outputs", ""] + [
            f"- {o['title']} (branch `{o.get('branch','')}`){' [has patch]' if o['has_patch'] else ''}"
            for o in slim["outputs"]
        ]
    (d / "summary.md").write_text("\n".join(lines) + "\n")

    processed += 1
    if processed % 25 == 0:
        print(f"  {processed}/{len(sessions)} sessions, {patches} patches...")

# ------------------------------------------------------------------
# 4. Recover setup scripts / test commands
# ------------------------------------------------------------------
for sid, meta in setup.items():
    d = OUT / "repos" / safe(sid)
    d.mkdir(parents=True, exist_ok=True)
    if meta.get("environmentSetupScript"):
        (d / "setup.sh").write_text(meta["environmentSetupScript"] + "\n")
    if meta.get("testCommand"):
        (d / "test-command.txt").write_text(meta["testCommand"] + "\n")

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
summary = {
    "repos": len(sources),
    "sessions": len(sessions),
    "sessions_processed": processed,
    "patches": patches,
    "repos_with_setup_or_test": len(setup),
    "error_count": len(errors),
    "errors": errors[:20],
    "note": (
        "User settings, environment variables, scheduled tasks and memories "
        "are currently unavailable (old $rpc endpoints return 404)."
    ),
}
dump(OUT / "export-summary.json", summary)

print("\n" + json.dumps(summary, indent=2))
print(f"\nDone → {OUT.resolve()}")

#Step 4: Map sources to repo for easy memory extractions

In [ ]:
"""Build owner/repo → github/<id> mapping from cached Jules sources."""
import json, pathlib

OUT = pathlib.Path("./jules-export")
sources_path = OUT / "sources.json"

if not sources_path.exists():
    raise FileNotFoundError(f"Expected sources.json to exist at {sources_path}")

sources = json.loads(sources_path.read_text())

mapping = {}
for s in sources:
    gh = (s.get("swebotSource") or {}).get("githubSource") or {}
    owner, name, static = gh.get("repoOwner"), gh.get("repoName"), gh.get("staticId")
    if owner and name and static:
        mapping[f"{owner}/{name}"] = static

(OUT / "repo_id_mapping.json").write_text(json.dumps(mapping, indent=2, sort_keys=True))
print(f"Mapped {len(mapping)} repos from cached sources → {OUT / 'repo_id_mapping.json'}")

#Step 5: Export memories

In [ ]:
#!/usr/bin/env python3
"""
Export Jules memories for every repo listed in source_mapping.json.

Fill in COOKIE / AT / F_SID / BL from DevTools while logged into
jules.google.com, then run:
    python3 export_jules_memories.py
"""
import json
import pathlib
import time
import urllib.parse
import urllib.request

# ============== FILL THESE FROM YOUR BROWSER DEV TOOL ==============
COOKIE = """PASTE_YOUR_COOKIE_HERE"""

AT     = "ALh...:1794...." # replace your at value
F_SID  = "45965....." # Replace with your sid
BL     = "boq_labs-language-aida-swebot-uiserver_20261106.00_p0"
# ==========================================================

MAPPING_FILE = pathlib.Path("./jules-export/repo_id_mapping.json")
OUT          = pathlib.Path("./jules-memories-export")
OUT.mkdir(exist_ok=True)

PAGE_SIZE    = 100
RPC_ID       = "c77bjc"
REQID_START  = 10_000_000          # any monotonic int; browser uses ~4M jumps
REQID_STEP   = 4_000_000


def load_source_ids() -> dict[str, str]:
    with MAPPING_FILE.open() as f:
        return json.load(f)


def _parse_batchexecute(raw: str, rpc_id: str) -> tuple:
    """Return (payload, cursor). Cursor is None on last page."""
    if raw.startswith(")]}'"):
        raw = raw[4:].lstrip("\n")

    needle = f'[["wrb.fr","{rpc_id}"'
    idx = raw.find(needle)
    if idx == -1:
        raise RuntimeError(
            f"{rpc_id!r} not found in response:\n{raw[:800]}"
        )

    arr, _ = json.JSONDecoder().raw_decode(raw, idx)   # [[wrb.fr-row]]
    row = next(
        (e for e in arr
         if isinstance(e, list) and len(e) >= 4
         and e[0] == "wrb.fr" and e[1] == rpc_id),
        None,
    )
    if row is None:
        raise RuntimeError(f"No wrb.fr row for {rpc_id}: {arr!r}")

    payload_str = row[2]
    cursor      = row[3] if len(row) > 3 else None
    if isinstance(cursor, str) and not cursor:
        cursor = None

    payload = json.loads(payload_str) if isinstance(payload_str, str) else payload_str
    return payload, cursor


def _find_records(obj):
    """Recursively locate the list of 5-tuple records inside the payload."""
    if isinstance(obj, list):
        if obj and all(
            isinstance(x, list) and len(x) >= 2
            and isinstance(x[0], str) and len(x[0]) == 36
            for x in obj
        ):
            return obj
        for item in obj:
            r = _find_records(item)
            if r is not None:
                return r
    return None


def batchexecute(rpc_id: str, rpc_args: list, reqid: int,
                 source_path: str) -> tuple:
    """Low-level call. Returns (payload, cursor)."""
    # Cursor goes INSIDE the args JSON as args[2]; tuple slot 3 stays null.
    f_req = json.dumps([[[rpc_id, json.dumps(rpc_args), None, "generic"]]])

    params = {
        "rpcids":      rpc_id,
        "source-path": source_path,
        "bl":          BL,
        "f.sid":       F_SID,
        "hl":          "en",
        "_reqid":      str(reqid),
        "rt":          "c",
    }
    url = ("https://jules.google.com/_/Swebot/data/batchexecute?"
           + urllib.parse.urlencode(params))

    headers = {
        "accept": "*/*",
        "content-type": "application/x-www-form-urlencoded;charset=utf-8",
        "origin": "https://jules.google.com",
        "referer": "https://jules.google.com/",
        "user-agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/157.0.0.0 Safari/537.36"),
        "cookie": COOKIE.strip(),
        "x-same-domain": "1",
        # If you get 401/403, copy the remaining x-browser-* headers
        # verbatim from a live DevTools request.
    }

    body = urllib.parse.urlencode({"f.req": f_req, "at": AT})
    req = urllib.request.Request(
        url, data=body.encode(), headers=headers, method="POST")
    with urllib.request.urlopen(req, timeout=45) as r:
        raw = r.read().decode()

    return _parse_batchexecute(raw, rpc_id)


def list_memories(source_id: str, source_path: str,
                  page_token: str | None, reqid: int) -> tuple:
    """Return (records, next_cursor)."""
    args = [source_id, PAGE_SIZE]
    if page_token:
        args.append(page_token)

    payload, cursor = batchexecute(RPC_ID, args, reqid, source_path)
    records = _find_records(payload) or []
    return records, cursor


def normalize(mem) -> dict:
    if not isinstance(mem, (list, tuple)) or len(mem) < 2:
        return {"raw": mem}
    return {
        "id":        mem[0],
        "content":   mem[1],
        "extra":     mem[2] if len(mem) > 2 else "",
        "timestamp": mem[3] if len(mem) > 3 else None,
        "flag":      mem[4] if len(mem) > 4 else None,
    }


def export_repo(repo_name: str, source_id: str, reqid_start: int) -> int:
    provider    = source_id.split("/")[0]              # e.g. "github"
    source_path = f"/repo/{provider}/{repo_name}/knowledge"

    print(f"\n=== {repo_name}  ({source_id}) ===")

    all_memories = []
    cursor = None
    page   = 0
    reqid  = reqid_start

    while True:
        page += 1
        print(f"  page {page}  cursor={'yes' if cursor else 'none'} ...",
              end=" ", flush=True)
        try:
            batch, cursor = list_memories(source_id, source_path, cursor, reqid)
        except Exception as e:
            print(f"ERROR: {e}")
            break

        print(f"{len(batch)} records")
        all_memories.extend(batch)

        if not batch or not cursor:
            break

        reqid += REQID_STEP
        time.sleep(0.4)

    safe     = repo_name.replace("/", "__")
    repo_dir = OUT / safe
    repo_dir.mkdir(exist_ok=True)

    (repo_dir / "memories_raw.json").write_text(
        json.dumps(all_memories, indent=2, default=str))

    clean = [normalize(m) for m in all_memories]
    (repo_dir / "memories.json").write_text(
        json.dumps(clean, indent=2, default=str))

    md = [f"# Memories for {repo_name} ({source_id})", "",
          f"Total: {len(clean)}", ""]
    for m in clean:
        ts = f" ({m['timestamp'][0]})" if isinstance(m.get("timestamp"), (list, tuple)) else ""
        md.append(f"## {m.get('id', '')}{ts}")
        md.append("")
        md.append(m.get("content") or "")
        md.append("")
    (repo_dir / "memories.md").write_text("\n".join(md))

    print(f"  → {repo_dir}  ({len(clean)} memories)")
    return len(clean)


def main():
    mapping = load_source_ids()
    print(f"Loaded {len(mapping)} repos from {MAPPING_FILE}")
    print(f"Output → {OUT.resolve()}\n")

    total = 0
    reqid = REQID_START
    for repo_name, source_id in mapping.items():
        total += export_repo(repo_name, source_id, reqid)
        reqid += REQID_STEP * 100     # keep reqid monotonic across repos

    print(f"\nDone. {total} memories across {len(mapping)} repos.")


if __name__ == "__main__":
    main()

#ALL done, You can mount drive and export to google drive